# M1 · Bag-of-words + feed-forward network

First model of the hierarchy of increasing multilingual capacity used in the paper.

| Model | Representation | Notebook |
|---|---|---|
| M1 | Bag-of-words + feed-forward network | `M1_bow.ipynb` |
| M2 | BiLSTM + frozen FastText vectors | `M2_bilstm_fasttext.ipynb` |
| M3 | mBERT (`bert-base-multilingual-cased`), full fine-tuning | `M3_mbert.ipynb` |
| M4 | XLM-RoBERTa (`xlm-roberta-base`), full fine-tuning | `M4_xlmr.ipynb` |

This notebook trains one model per source corpus and seed (42, 43, 44), evaluates it on the test
partition of all six corpora, and saves the predicted probabilities of every cell to `.npz`.
It also produces the NLLB-200 translations used by M1 and M2 (translate-test and translate-train).

| Code | Corpus | Language | Platform | n |
|---|---|---|---|---|
| PT1 | HateBR (Vargas et al., 2022) | Portuguese | Instagram, political accounts | 7,000 |
| PT2 | OffComBR-3 (de Pelle & Moreira, 2017) | Portuguese | news portal comments | 1,033 |
| PT3 | ToLD-Br (Leite et al., 2020), stratified subsample | Portuguese | Twitter, general | 7,000 |
| EN | Davidson et al. (2017) | English | Twitter, general | 24,783 |
| ID | Alfina et al. (2017) | Indonesian | Twitter, political | 713 |
| KZ | Bolatbek et al. (2025) | Kazakh | YouTube, Telegram, VK | 10,190 |

**Inputs.** `protocolo_pibic.py` and the `splits/` folder in `BASE`, and the six corpus files in `BASE/dados/`
(see `data/corpora/README.md`). On Colab, `BASE` is `MyDrive/pibic`; elsewhere it is `./pibic`.

## 1. Imports and configuration

In [ ]:
import os, re, string, json, csv, datetime
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOPWORDS = {
    'pt': set(stopwords.words('portuguese')),
    'en': set(stopwords.words('english')),
    'id': set(stopwords.words('indonesian')),
    'kz': set(),  # NLTK has no Kazakh stopword list
}

SEED = 42
np.random.seed(SEED)

# ── M1 hyperparameters (fixed; never tuned against the test set) ────────────
HIDDEN    = 100
EPOCHS    = 60
PATIENCE  = 7
LR        = 0.001
BATCH     = 32
DROPOUT   = 0.2
POS_W_MAX = 10.0   # cap on pos_weight
THETA     = 2      # minimum vocabulary frequency
MIN_VAL_CALIB = 100  # below this size, threshold calibration is unstable -> use 0.5

print("Imports OK — M1")

In [ ]:
import time
from contextlib import contextmanager

TEMPOS = {}
_T0_NOTEBOOK = time.time()

def _fmt(s):
    m, s = divmod(s, 60); h, m = divmod(m, 60)
    if h: return f"{int(h)}h{int(m):02d}m{s:04.1f}s"
    if m: return f"{int(m)}m{s:04.1f}s"
    return f"{s:.1f}s"

@contextmanager
def cronometro(nome):
    '''Usage: with cronometro("section name"): ...block...'''
    t0 = time.time()
    print(f"⏱  [{nome}] starting...")
    yield
    dt = time.time() - t0
    TEMPOS[nome] = dt
    total = time.time() - _T0_NOTEBOOK
    print(f"⏱  [{nome}] done in {_fmt(dt)}  (notebook total: {_fmt(total)})")

print("Timer ready — model: M1 bag-of-words")


## 2. Machine translation settings

`facebook/nllb-200-distilled-600M`, greedy decoding (`num_beams=1`), maximum length 256 tokens on
input and output. Translations are cached on disk and reused by M2.

In [ ]:
!pip install -q transformers sentencepiece  # needed on Colab

MT_CHECKPOINT = 'facebook/nllb-200-distilled-600M'
NUM_BEAMS     = 1
MAX_LEN_IN    = 256
MAX_LEN_OUT   = 256
BATCH_MT      = 64

LANG_CODES = {'pt': 'por_Latn', 'en': 'eng_Latn', 'id': 'ind_Latn', 'kz': 'kaz_Cyrl'}

# Translation cache on Drive, so it persists across sessions:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/pibic/traducoes'
except ImportError:
    CACHE_DIR = os.path.abspath('./pibic/traducoes')
os.makedirs(CACHE_DIR, exist_ok=True)

REGISTRO_MT = {
    'checkpoint': MT_CHECKPOINT, 'num_beams': NUM_BEAMS,
    'max_length': MAX_LEN_OUT, 'data_traducao': str(datetime.date.today()),
}
REGISTRO_M1V2 = {
    'hidden': HIDDEN, 'epochs': EPOCHS, 'patience': PATIENCE,
    'bow': 'log(1+tf) + L2', 'pos_weight': f'n_neg/n_pos (teto {POS_W_MAX})',
    'threshold': 'calibrado na validação (macro-F1)',
}
print(REGISTRO_MT); print(REGISTRO_M1V2)

## 3. Loading and binarising the six corpora

Positive class = problematic content. PT1: `label_final`. PT2: `yes`. PT3: any category marked by at
least 2 of 3 annotators. EN: `class != 2` (hate or offensive). ID: `HS`. KZ: `label != 0`.

In [ ]:
# ═══════════════ PROTOCOL SHARED BY THE FOUR MODELS ═══════════════
# Loads protocolo_pibic.py, which holds the ToLD-Br (PT3) loader, the shared
# partitions, prediction persistence and metrics. Place the file in BASE
# before running.
import os, sys, json
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/MyDrive/pibic'
except Exception:
    BASE = os.path.abspath('./pibic')

DATA_DIR  = os.path.join(BASE, 'dados')
OUT_DIR   = os.path.join(BASE, 'resultados')
SPLIT_DIR = os.path.join(BASE, 'splits')
for d in (DATA_DIR, OUT_DIR, SPLIT_DIR):
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, BASE)
import protocolo_pibic as PP

def caminho_dado(nome):
    # Accepts the file in the data directory or in the current directory.
    for c in (os.path.join(DATA_DIR, nome), nome):
        if os.path.exists(c):
            return c
    raise FileNotFoundError(
        f'{nome} not found in {DATA_DIR} or in the current directory.')

TAG = datetime.now().strftime('%Y%m%d_%H%M')
print(f'BASE = {BASE}\nTAG  = {TAG}')


def parse_offcombr3(path):
    texts, labels = [], []
    with open(path, encoding='utf-8', errors='ignore') as f:
        in_data = False
        for line in f:
            line = line.strip()
            if not in_data:
                in_data = line.lower().startswith('@data')
                continue
            if not line:
                continue
            lab, _, resto = line.partition(',')
            texto = resto.strip().strip("'")
            if lab in ('yes', 'no') and texto:
                texts.append(texto)
                labels.append(1 if lab == 'yes' else 0)
    return texts, labels


# ═══════════════ SIX CORPORA ═══════════════
#
# Three Portuguese corpora give six ordered Portuguese→Portuguese cells, so the
# domain cost does not rest on a single pair of corpora.
#
# Leite, Silva, Bontcheva & Scarton (2020), AACL-IJCNLP. License: CC BY-SA 4.0.
# Download ToLD-BR.csv from https://github.com/joaoaleite/ToLD-Br into DATA_DIR.

LIMIAR_ANOTADORES_PT3 = 2      # simple majority; see protocolo_pibic.carregar_toldbr
N_MAX_PT3 = 7000               # matches the size of HateBR; None keeps all 21,000

DATASETS = {}

df = pd.read_csv(caminho_dado('HateBR.csv'))
DATASETS['PT1'] = {'texts': [PP.sanitizar(t) for t in df['comentario'].astype(str)],
                   'labels': df['label_final'].astype(int).tolist(),
                   'lang': 'pt', 'nome': 'HateBR (PT1)',
                   'dominio': 'político / Instagram'}

t, l = parse_offcombr3(caminho_dado('OffComBR3.arff'))
DATASETS['PT2'] = {'texts': [PP.sanitizar(x) for x in t], 'labels': l,
                   'lang': 'pt', 'nome': 'OffComBR-3 (PT2)',
                   'dominio': 'comentários de portal de notícias'}

t, l, META_PT3 = PP.carregar_toldbr(caminho_dado('ToLD-BR.csv'),
                                    limiar_anotadores=LIMIAR_ANOTADORES_PT3,
                                    n_max=N_MAX_PT3)
DATASETS['PT3'] = {'texts': t, 'labels': l,
                   'lang': 'pt', 'nome': 'ToLD-Br (PT3)',
                   'dominio': 'Twitter, geral', 'meta': META_PT3}

df = pd.read_csv(caminho_dado('DataIngles.csv'))
DATASETS['EN'] = {'texts': [PP.sanitizar(x) for x in df['tweet'].astype(str)],
                  'labels': (df['class'] != 2).astype(int).tolist(),
                  'lang': 'en', 'nome': 'Davidson (EN)', 'dominio': 'Twitter, geral'}

bruto = open(caminho_dado('DataIndonesio.txt'), encoding='utf-8',
             errors='ignore').read().replace('\r\n', '\n').replace('\r', '\n')
linhas = [x for x in bruto.split('\n') if '\t' in x][1:]
DATASETS['ID'] = {'texts': [PP.sanitizar(x.split('\t', 1)[1]) for x in linhas],
                  'labels': [1 if x.split('\t', 1)[0].strip() == 'HS' else 0
                             for x in linhas],
                  'lang': 'id', 'nome': 'Alfina (ID)', 'dominio': 'Twitter, político'}

df = pd.read_csv(caminho_dado('DataKazake.csv'))
DATASETS['KZ'] = {'texts': [PP.sanitizar(x) for x in df['message'].astype(str)],
                  'labels': (df['label'] != 0).astype(int).tolist(),
                  'lang': 'kz', 'nome': 'Bolatbek (KZ)',
                  'dominio': 'YouTube / Telegram / VK'}

SIGLAS = [s for s in PP.SIGLAS if s in DATASETS]
for k in SIGLAS:
    ds = DATASETS[k]
    ds['sigla'] = k
    ds['prev_pos'] = float(np.mean(ds['labels']))
    ds['baseline'] = PP.baseline_majoritaria(ds['labels'])

# Trivial floor per corpus (macro-F1 of the majority-class classifier).
BASELINE_TRIVIAL = {k: DATASETS[k]['baseline'] for k in SIGLAS}

print(f"\n{'Code':<6} {'Corpus':<20} {'Lang':<5} {'N':>7} {'%pos':>7} {'trivial':>9}")
print('-' * 60)
for k in SIGLAS:
    ds = DATASETS[k]
    print(f"{k:<6} {ds['nome']:<20} {ds['lang']:<5} {len(ds['labels']):>7,} "
          f"{ds['prev_pos']*100:>6.1f}% {ds['baseline']:>9.3f}")

# PT2 (19.6%) and PT3 (19.3%) have almost the same prevalence, so the PT2 x PT3
# pair is a domain contrast without a base-rate confounder.


### 3.1 Trivial floor (majority class)

Macro-F1 of a classifier that always predicts the majority class of each corpus.

In [ ]:
print(f"{'Test':<16} {'maj.':>4} {'Trivial macro-F1':>17}")
BASELINE_TRIVIAL = {}
for k, ds in DATASETS.items():
    y = np.array(ds['labels'])
    maj = int(y.mean() >= 0.5)
    mf1 = f1_score(y, np.full_like(y, maj), average='macro', zero_division=0)
    BASELINE_TRIVIAL[k] = mf1
    print(f"{ds['nome']:<16} {maj:>4} {mf1:>17.3f}")

## 4. Shared partitions

The partitions are loaded from `splits/`, so the four models see exactly the same test sets.

In [ ]:

# ═══════════════ PARTITIONS SHARED BY M1–M4 ═══════════════
# Loaded from SPLIT_DIR. If each notebook drew its own split, differences between
# models would mix model effects with partition effects.
CHAVES = {k: [PP.chave_norm(t) for t in DATASETS[k]['texts']] for k in SIGLAS}

for k in SIGLAS:
    ds = DATASETS[k]
    outros = {o: CHAVES[o] for o in SIGLAS if o != k}
    ds['idx_tr'], ds['idx_val'], ds['idx_te'] = PP.obter_particoes(
        k, ds['texts'], ds['labels'], SPLIT_DIR, chaves_outros_corpora=outros)

PP.conferir_particoes(SIGLAS, SPLIT_DIR)

print(f"\n{'Corpus':<6} {'train':>8} {'val':>7} {'test':>7} {'%pos test':>12}")
print('-' * 44)
for k in SIGLAS:
    ds = DATASETS[k]
    y = np.array(ds['labels'])
    print(f"{k:<6} {len(ds['idx_tr']):>8,} {len(ds['idx_val']):>7,} "
          f"{len(ds['idx_te']):>7,} {y[ds['idx_te']].mean()*100:>11.1f}%")


In [ ]:

# ═══════════════ RESULTS AND PREDICTIONS LOG ═══════════════
# Every cell of the matrix saves the predicted PROBABILITIES to .npz, so AUC and
# the three threshold regimes (R1 fixed 0.5, R2 prevalence-matched, R3 oracle)
# can be recomputed without retraining.
MODELO_ID = 'M1'
PRED_DIR = PP.dir_predicoes(OUT_DIR, MODELO_ID, TAG)
RESULTADOS = []

def registrar_celula(k_tr, k_te, cenario, semente, p, y, limiar_politica=0.5,
                     verboso=True):
    PP.salvar_predicoes(PRED_DIR, k_tr, k_te, semente, p, y, cenario=cenario)
    m   = PP.metricas(y, p, limiar_politica)
    m05 = PP.metricas(y, p, 0.5)
    reg = PP.tres_regimes(y, p, DATASETS[k_te]['prev_pos'])
    linha = dict(modelo=MODELO_ID, treino=k_tr, teste=k_te, cenario=cenario,
                 semente=semente, grupo=PP.grupo_cenario(k_tr, k_te),
                 mesma_lingua=(DATASETS[k_tr]['lang'] == DATASETS[k_te]['lang']),
                 n_teste=int(len(y)), piso=DATASETS[k_te]['baseline'],
                 macro_f1=m['macro_f1'], limiar=m['limiar'],
                 macro_f1_05=m05['macro_f1'], auc_roc=m05['auc_roc'],
                 auc_pr=m05['auc_pr'], f1_pos_05=m05['f1_pos'],
                 f1_neg_05=m05['f1_neg'], rec_pos_05=m05['rec_pos'],
                 taxa_predicao_positiva_05=m05['taxa_predicao_positiva'])
    linha.update(reg)
    linha['R2_sobre_piso'] = linha['R2_prevalencia'] - linha['piso']
    RESULTADOS.append(linha)
    if verboso:
        marca = 'ok' if linha['R2_sobre_piso'] > 0 else 'BELOW FLOOR'
        print(f"    {k_tr:>4} -> {k_te:<4} [{cenario:<12}] s{semente}  "
              f"R1={linha['macro_f1_05']:.3f}  R2={linha['R2_prevalencia']:.3f}  "
              f"AUC={linha['auc_roc']:.3f}  {marca}")
    return linha


## 5. NLLB-200 translation module (disk cache)

Translates every corpus into every other language of the matrix. A full first run takes several
hours on a single GPU; later runs read the cache.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_MT = {'tok': None, 'model': None}

def carregar_nllb():
    if _MT['model'] is None:
        print(f"Loading {MT_CHECKPOINT}...")
        _MT['tok']   = AutoTokenizer.from_pretrained(MT_CHECKPOINT)
        _MT['model'] = AutoModelForSeq2SeqLM.from_pretrained(MT_CHECKPOINT)
        _MT['model'].to('cuda' if torch.cuda.is_available() else 'cpu').eval()
    return _MT['tok'], _MT['model']

def traduzir_lote(textos, src_lang, tgt_lang):
    tok, model = carregar_nllb()
    tok.src_lang = LANG_CODES[src_lang]
    device = next(model.parameters()).device
    bos_id = tok.convert_tokens_to_ids(LANG_CODES[tgt_lang])
    saidas = []
    for i in range(0, len(textos), BATCH_MT):
        lote = [str(t)[:2000] for t in textos[i:i+BATCH_MT]]
        enc  = tok(lote, return_tensors='pt', padding=True,
                   truncation=True, max_length=MAX_LEN_IN).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=bos_id,
                                 num_beams=NUM_BEAMS, max_length=MAX_LEN_OUT)
        saidas.extend(tok.batch_decode(gen, skip_special_tokens=True))
        if (i // BATCH_MT) % 20 == 0:
            print(f"    {i+len(lote):,}/{len(textos):,}")
    return saidas

def obter_traducao(key_dataset, tgt_lang):
    ds = DATASETS[key_dataset]
    assert ds['lang'] != tgt_lang
    path = os.path.join(CACHE_DIR, f"{key_dataset}__para_{tgt_lang}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        assert len(df) == len(ds['texts']), f"cache inconsistente: {path}"
        return df['traduzido'].astype(str).tolist()
    print(f"Translating {ds['nome']} → {tgt_lang.upper()} ({len(ds['texts']):,} docs)...")
    trad = traduzir_lote(ds['texts'], ds['lang'], tgt_lang)
    pd.DataFrame({'idx': range(len(trad)), 'original': ds['texts'],
                  'traduzido': trad}).to_csv(path, index=False)
    print(f"  → saved to {path}")
    return trad

LANGS = sorted({ds['lang'] for ds in DATASETS.values()})
DIRECOES = [(k, lg) for k in DATASETS for lg in LANGS if DATASETS[k]['lang'] != lg]
print(f"{len(DIRECOES)} corpus translations needed (cached ones are skipped):")
for k, lg in DIRECOES:
    feito = os.path.exists(os.path.join(CACHE_DIR, f"{k}__para_{lg}.csv"))
    print(f"  {'[ok]' if feito else '[  ]'} {DATASETS[k]['nome']:<16} → {lg.upper()}")

with cronometro("traducao_nllb"):
    for k, lg in DIRECOES:
        obter_traducao(k, lg)

### 5.1 Spot check of translations

In [ ]:
def amostrar_traducoes(key_dataset, tgt_lang, n=30, seed=SEED):
    path = os.path.join(CACHE_DIR, f"{key_dataset}__para_{tgt_lang}.csv")
    df = pd.read_csv(path).sample(n, random_state=seed)
    for _, r in df.iterrows():
        print(f"[{r['idx']}] ORIG: {str(r['original'])[:150]}")
        print(f"      TRAD: {str(r['traduzido'])[:150]}\n")

# Example: amostrar_traducoes('EN', 'pt')

## 6. Preprocessing and bag-of-words

URLs, mentions and hashtags removed, lower case, punctuation removed, NLTK tokenisation and
stopwords, tokens with less than 70% alphabetic characters or shorter than two characters dropped.
Features: `log(1+tf)` with L2 normalisation per document; vocabulary from the training partition
only, minimum frequency 2.

In [ ]:
def preprocess(text, lang='pt'):
    stop = STOPWORDS.get(lang, set())
    tok_lang = {'pt':'portuguese','en':'english','id':'indonesian'}.get(lang,'english')
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', str(text))
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    try:
        tokens = word_tokenize(text, language=tok_lang)
    except:
        tokens = text.split()
    tokens = [t for t in tokens if t not in stop]
    tokens = [t for t in tokens if t and sum(c.isalpha() for c in t)/len(t) >= 0.7]
    tokens = [t for t in tokens if len(t) >= 2]
    return tokens

def build_vocab(tokenized_list):
    counts = Counter(t for doc in tokenized_list for t in doc)
    vocab  = {w for w, c in counts.items() if c >= THETA}
    return {w: i+1 for i, w in enumerate(sorted(vocab))}

def make_X(tokenized, w2i):
    '''Bag-of-words: log(1+tf) per term + L2 normalisation per document.'''
    X = np.zeros((len(tokenized), len(w2i)+1), dtype=np.float32)
    for i, doc in enumerate(tokenized):
        for t in doc:
            if t in w2i:
                X[i, w2i[t]] += 1
    X = np.log1p(X)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

for key, ds in DATASETS.items():
    ds['tokenized'] = [preprocess(t, ds['lang']) for t in ds['texts']]
print("Preprocessing of the original texts done.")

## 7. Network: class-weighted BCE and validation-calibrated threshold

One hidden layer of 100 ReLU units, dropout 0.2, Adam (learning rate 0.001), batch 32, up to 60
epochs with early stopping (patience 7) on validation loss. `pos_weight = n_neg / n_pos`, capped at 10.

In [ ]:
def relu(z):    return np.maximum(0, z)
def relu_d(z):  return (z > 0).astype(np.float32)
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def bce_w(yt, yp, w_pos, w_neg):
    yp = np.clip(yp, 1e-7, 1-1e-7)
    return -np.mean(w_pos*yt*np.log(yp) + w_neg*(1-yt)*np.log(1-yp))

def forward(X, W1, b1, W2, b2, training=True, dr=DROPOUT):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    mask = None
    if training:
        mask = (np.random.rand(*a1.shape) > dr).astype(np.float32)
        a1   = a1 * mask / (1 - dr)
    a2 = sigmoid(a1 @ W2 + b2)
    return z1, a1, mask, None, a2

def adam(p, g, m, v, t, lr=LR, b1=0.9, b2=0.999, eps=1e-8):
    m = b1*m + (1-b1)*g
    v = b2*v + (1-b2)*g**2
    p -= lr * (m/(1-b1**t)) / (np.sqrt(v/(1-b2**t)) + eps)
    return p, m, v

def treinar(X_tr, y_tr, X_val, y_val,
            hidden=HIDDEN, dr=DROPOUT, lr=LR, batch=BATCH,
            epochs=EPOCHS, patience=PATIENCE, seed=SEED, verbose=True):
    np.random.seed(seed)
    n_pos = max(int(y_tr.sum()), 1)
    n_neg = max(int(len(y_tr) - y_tr.sum()), 1)
    w_pos = float(min(n_neg / n_pos, POS_W_MAX))
    w_neg = 1.0
    if verbose:
        print(f"    pos_weight = {w_pos:.2f} (pos={n_pos:,}, neg={n_neg:,})")

    inp = X_tr.shape[1]
    W1 = np.random.randn(inp, hidden).astype(np.float32) * np.sqrt(2/inp)
    b1 = np.zeros((1, hidden), dtype=np.float32)
    W2 = np.random.randn(hidden, 1).astype(np.float32) * np.sqrt(2/hidden)
    b2 = np.zeros((1, 1), dtype=np.float32)
    mW1,vW1 = np.zeros_like(W1),np.zeros_like(W1)
    mb1,vb1 = np.zeros_like(b1),np.zeros_like(b1)
    mW2,vW2 = np.zeros_like(W2),np.zeros_like(W2)
    mb2,vb2 = np.zeros_like(b2),np.zeros_like(b2)
    best_loss, pat, best_w, t_adam = np.inf, 0, None, 0
    n = X_tr.shape[0]

    for epoch in range(1, epochs+1):
        perm = np.random.permutation(n)
        Xs, ys = X_tr[perm], y_tr[perm]
        for b in range(int(np.ceil(n/batch))):
            Xb = Xs[b*batch:(b+1)*batch]
            yb = ys[b*batch:(b+1)*batch].reshape(-1,1)
            t_adam += 1
            z1,a1,mask,_,a2 = forward(Xb,W1,b1,W2,b2,True,dr)
            # gradient of the weighted BCE: dz = (w_pos*y + w_neg*(1-y)) * (p - y)
            wvec = w_pos*yb + w_neg*(1-yb)
            dz2 = wvec*(a2-yb)/len(yb)
            dW2 = a1.T@dz2; db2 = dz2.sum(0,keepdims=True)
            da1 = (dz2@W2.T)*mask/(1-dr)
            dz1 = da1*relu_d(z1)
            dW1 = Xb.T@dz1; db1 = dz1.sum(0,keepdims=True)
            W2,mW2,vW2 = adam(W2,dW2,mW2,vW2,t_adam)
            b2,mb2,vb2 = adam(b2,db2,mb2,vb2,t_adam)
            W1,mW1,vW1 = adam(W1,dW1,mW1,vW1,t_adam)
            b1,mb1,vb1 = adam(b1,db1,mb1,vb1,t_adam)

        _,_,_,_,vo = forward(X_val,W1,b1,W2,b2,False,dr)
        vl = bce_w(y_val.reshape(-1,1), vo, w_pos, w_neg)
        if vl < best_loss:
            best_loss = vl; pat = 0
            best_w = (W1.copy(),b1.copy(),W2.copy(),b2.copy())
        else:
            pat += 1
            if pat >= patience:
                if verbose: print(f"    Early stopping at epoch {epoch}")
                break
    return best_w

def melhor_threshold(pesos, X_val, y_val, min_n=MIN_VAL_CALIB):
    '''Threshold that maximises macro-F1 on the VALIDATION set (never on test).
    Requires at least min_n validation examples, otherwise returns 0.5.
    Used only for in-corpus evaluation. Transfer cells use TH_CROSS = 0.5,
    because a threshold calibrated on the source class balance does not carry
    over to the target.'''
    if len(y_val) < min_n:
        return 0.5
    _,_,_,_,p = forward(X_val, *pesos, False)
    p = p.flatten()
    ths = np.arange(0.10, 0.91, 0.02)
    f1s = [f1_score(y_val, (p >= t).astype(int), average='macro',
                    zero_division=0) for t in ths]
    return float(ths[int(np.argmax(f1s))])

TH_CROSS = 0.5  # fixed threshold for every transfer cell (cross-domain, cross-lingual, MT, TT)

def prever(X, pesos, threshold=0.5):
    _,_,_,_,out = forward(X, *pesos, False)
    return (out.flatten() >= threshold).astype(int)

def avaliar(y_true, y_pred, titulo):
    mf1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    m = {'acc':  accuracy_score(y_true, y_pred),
         'prec': precision_score(y_true, y_pred, zero_division=0),
         'rec':  recall_score(y_true, y_pred, zero_division=0),
         'f1':   f1_score(y_true, y_pred, zero_division=0),
         'macro_f1': mf1,
         'cm':   confusion_matrix(y_true, y_pred).tolist()}
    print(f"  {titulo:<48} Macro-F1: {mf1:.3f}")
    return m

print("Network defined.")

## 8. Base training: one model per corpus and seed

In [ ]:
SEMENTES = [42, 43, 44]

def probs_m1(X, pesos):
    # Keep the probabilities: they are needed for AUC and the threshold regimes.
    _, _, _, _, out = forward(X, *pesos, False)
    return out.flatten().astype(np.float64)

MODELOS = {}      # (corpus, seed) -> dict

with cronometro("treino_modelos_base"):
    for semente in SEMENTES:
        for k in SIGLAS:
            ds = DATASETS[k]
            print(f"\n=== M1 base: {ds['nome']} · seed {semente} ===")
            tok = ds['tokenized']
            y = np.array(ds['labels'], dtype=np.float32)
            w2i = build_vocab([tok[i] for i in ds['idx_tr']])
            X_tr = make_X([tok[i] for i in ds['idx_tr']], w2i)
            X_val = make_X([tok[i] for i in ds['idx_val']], w2i)
            y_val = y[ds['idx_val']]
            print(f"  train={X_tr.shape[0]:,} | val={X_val.shape[0]:,} | vocab={len(w2i):,}")
            pesos = treinar(X_tr, y[ds['idx_tr']], X_val, y_val,
                            seed=semente, verbose=False)
            th = melhor_threshold(pesos, X_val, y_val.astype(int))
            MODELOS[(k, semente)] = {'pesos': pesos, 'w2i': w2i, 'th_intra': th,
                                     'p_val': probs_m1(X_val, pesos),
                                     'y_val': y_val.astype(int)}
            print(f"  in-corpus threshold calibrated (val, n={len(y_val)}): {th:.2f}")
print("\nBase models trained.")

## 9. In-corpus, cross-domain and cross-lingual cells (zero-shot)

In [ ]:
with cronometro("intra_e_cross"):
    for semente in SEMENTES:
        print(f"\n─── seed {semente} ───")
        for k_tr in SIGLAS:
            m = MODELOS[(k_tr, semente)]
            for k_te in SIGLAS:
                ds_te = DATASETS[k_te]
                # ALWAYS the test partition, as in the diagonal and in M3/M4.
                tok_te = [ds_te['tokenized'][i] for i in ds_te['idx_te']]
                y_te = np.array(ds_te['labels'])[ds_te['idx_te']]
                X_te = make_X(tok_te, m['w2i'])
                p = probs_m1(X_te, m['pesos'])
                # The `macro_f1` column keeps the training-time threshold policy (calibrated
                # only on the diagonal); the R1/R2/R3 columns do not depend on it.
                lim = m['th_intra'] if k_tr == k_te else 0.5
                registrar_celula(k_tr, k_te, 'base', semente, p, y_te, lim)

## 10. Translate-test

The test set of the target is translated into the language of the source and scored by the base model.

In [ ]:
def executar_mt(k_tr, k_te, semente):
    ds_tr, ds_te = DATASETS[k_tr], DATASETS[k_te]
    m = MODELOS[(k_tr, semente)]
    trad = obter_traducao(k_te, ds_tr['lang'])          # aligned with the whole corpus
    idx = ds_te['idx_te']                                # ...but evaluated on the test partition only
    tok_trad = [preprocess(trad[i], ds_tr['lang']) for i in idx]
    X_te = make_X(tok_trad, m['w2i'])
    y_te = np.array(ds_te['labels'])[idx]
    registrar_celula(k_tr, k_te, 'mt', semente, probs_m1(X_te, m['pesos']), y_te, 0.5)

with cronometro("translate_test_mt"):
    for semente in SEMENTES:
        print(f"\n─── MT · seed {semente} ───")
        for k_tr in SIGLAS:
            for k_te in SIGLAS:
                if k_tr != k_te and DATASETS[k_tr]['lang'] != DATASETS[k_te]['lang']:
                    executar_mt(k_tr, k_te, semente)

## 11. Translate-train

The training set of the source is translated into the language of the target and a new model is
trained for each cell.

In [ ]:
def executar_tt(k_tr, k_te, semente):
    ds_tr, ds_te = DATASETS[k_tr], DATASETS[k_te]
    lang_alvo = ds_te['lang']
    trad = obter_traducao(k_tr, lang_alvo)
    tok_trad = [preprocess(t, lang_alvo) for t in trad]
    y = np.array(ds_tr['labels'], dtype=np.float32)

    w2i = build_vocab([tok_trad[i] for i in ds_tr['idx_tr']])
    X_tr = make_X([tok_trad[i] for i in ds_tr['idx_tr']], w2i)
    X_val = make_X([tok_trad[i] for i in ds_tr['idx_val']], w2i)
    y_val = y[ds_tr['idx_val']]
    pesos = treinar(X_tr, y[ds_tr['idx_tr']], X_val, y_val,
                    seed=semente, verbose=False)

    idx = ds_te['idx_te']
    X_te = make_X([ds_te['tokenized'][i] for i in idx], w2i)
    y_te = np.array(ds_te['labels'])[idx]
    registrar_celula(k_tr, k_te, 'tt', semente, probs_m1(X_te, pesos), y_te, 0.5)

EXECUTAR_TT = True
with cronometro("translate_train_tt"):
    if EXECUTAR_TT:
        for semente in SEMENTES:
            print(f"\n─── TT · seed {semente} ───")
            for k_tr in SIGLAS:
                for k_te in SIGLAS:
                    if k_tr != k_te and DATASETS[k_tr]['lang'] != DATASETS[k_te]['lang']:
                        executar_tt(k_tr, k_te, semente)

## 12. Summary: run time and translation tables

In [ ]:
print("\n" + "="*50)
print("RUN TIME SUMMARY")
print("="*50)
for nome, dt in TEMPOS.items():
    print(f"  {nome:<28} {_fmt(dt)}")
print("-"*50)
print(f"  {'TOTAL (notebook)':<28} {_fmt(time.time()-_T0_NOTEBOOK)}")


In [ ]:

# ═══════════════ TRANSLATE-TEST x TRANSLATE-TRAIN ═══════════════
# Scenario names: 'base' (zero-shot), 'mt' (translate-test), 'tt' (translate-train).
# Each value is the MEAN over the three seeds. R2 (prevalence-matched threshold)
# is the main column; R1 (threshold 0.5) is shown next to it.

def pegar(k_tr, k_te, cen, col='R2_prevalencia'):
    r = [x for x in RESULTADOS
         if x['treino'] == k_tr and x['teste'] == k_te and x['cenario'] == cen]
    return float(np.mean([x[col] for x in r])) if r else np.nan

fmt = lambda v: f'{v:7.3f}' if not np.isnan(v) else '      —'

for col, rot in [('R1_fixo', 'R1 · fixed threshold 0.5'),
                 ('R2_prevalencia', 'R2 · prevalence-matched  ← main')]:
    print(f'\n{"="*72}\n  {rot}\n{"="*72}')
    print(f"{'Train':<7}{'Test':<7}{'Trivial':>8}{'zero-shot':>11}"
          f"{'MT':>8}{'TT':>8}{'Δ MT':>8}{'Δ TT':>8}")
    print('-' * 72)
    for k_tr in SIGLAS:
        for k_te in SIGLAS:
            if k_tr == k_te or DATASETS[k_tr]['lang'] == DATASETS[k_te]['lang']:
                continue
            base = pegar(k_tr, k_te, 'base', col)
            mt   = pegar(k_tr, k_te, 'mt',   col)
            tt   = pegar(k_tr, k_te, 'tt',   col)
            print(f'{k_tr:<7}{k_te:<7}{BASELINE_TRIVIAL[k_te]:>8.3f}'
                  f'{fmt(base)}{fmt(mt)}{fmt(tt)}{fmt(mt-base)}{fmt(tt-base)}')

print('\nPositive Δ = translation helps relative to direct transfer.')
print('Always compare against the trivial floor of the TARGET: a gain over zero-shot')
print('means nothing if the whole cell is below the floor.')


## 13. Results persistence (same format as M3/M4)

In [ ]:

# ═══════════════ PERSISTENCE (same format as M3/M4) ═══════════════
DF = pd.DataFrame(RESULTADOS)

AGG = (DF[DF.cenario == 'base']
       .groupby(['treino', 'teste', 'grupo', 'mesma_lingua', 'piso', 'n_teste'],
                as_index=False)
       .agg(macro_f1_05_media=('macro_f1_05', 'mean'),
            macro_f1_05_dp=('macro_f1_05', 'std'),
            R1_fixo=('R1_fixo', 'mean'),
            R2_prevalencia=('R2_prevalencia', 'mean'),
            R2_dp=('R2_prevalencia', 'std'),
            R3_oraculo=('R3_oraculo', 'mean'),
            custo_calibracao=('custo_calibracao', 'mean'),
            auc_media=('auc_roc', 'mean'), auc_dp=('auc_roc', 'std'),
            n_sementes=('semente', 'nunique')))
AGG['R2_sobre_piso'] = AGG['R2_prevalencia'] - AGG['piso']

# Sanity check: R3 is an upper bound and must dominate R1 and R2 in every cell.
viol = AGG[(AGG.R3_oraculo < AGG.R1_fixo - 1e-6) |
           (AGG.R3_oraculo < AGG.R2_prevalencia - 1e-6)]
print(f'Violations of "R3 is an upper bound": {len(viol)} of {len(AGG)}')

print('\n── Aggregated matrix (regime R2) ──')
print(AGG.sort_values(['treino', 'teste'])[
    ['treino', 'teste', 'grupo', 'piso', 'R1_fixo', 'R2_prevalencia',
     'R3_oraculo', 'R2_sobre_piso', 'auc_media', 'macro_f1_05_dp']
    ].to_string(index=False, float_format=lambda v: f'{v:6.3f}'))

print('\n── Means by group ──')
print(AGG.groupby('grupo')[['R1_fixo', 'R2_prevalencia', 'R3_oraculo',
                            'custo_calibracao', 'auc_media']]
        .agg(['mean', 'count']).round(3).to_string())

SAIDA = dict(
    modelo=MODELO_ID, gerado_em=datetime.now().isoformat(),
    corpora={k: {kk: vv for kk, vv in DATASETS[k].items()
                 if kk not in ('texts', 'labels', 'tokenized',
                               'idx_tr', 'idx_val', 'idx_te')}
             for k in SIGLAS},
    particoes={k: dict(n_treino=len(DATASETS[k]['idx_tr']),
                       n_val=len(DATASETS[k]['idx_val']),
                       n_teste=len(DATASETS[k]['idx_te'])) for k in SIGLAS},
    resultados=RESULTADOS,
    agregado=AGG.to_dict(orient='records'),
    tempos_execucao={k: round(v / 60, 3) for k, v in sorted(TEMPOS.items())}
        if 'TEMPOS' in dir() else {},
)
arq_json = os.path.join(OUT_DIR, f'{MODELO_ID}_resultados_{TAG}.json')
with open(arq_json, 'w', encoding='utf-8') as f:
    json.dump(SAIDA, f, ensure_ascii=False, indent=2, default=float)
AGG.to_csv(os.path.join(OUT_DIR, f'{MODELO_ID}_matriz_{TAG}.csv'), index=False)

print(f'\nSaved:\n  {arq_json}\n  {MODELO_ID}_matriz_{TAG}.csv\n  {PRED_DIR}/')
